# Day 3 模块 4：用 sklearn 训练随机森林

现在把概念落成代码。


In [1]:
from pathlib import Path
import pandas as pd

candidate_paths = [
    Path('day3_cafe_sales.csv'),
    Path('day3') / 'day3_cafe_sales.csv',
    Path('教学课程') / 'day3' / 'day3_cafe_sales.csv',
]
for path in candidate_paths:
    if path.exists():
        csv_path = path
        break
else:
    raise FileNotFoundError('未找到 day3_cafe_sales.csv')

df = pd.read_csv(csv_path)
print(csv_path.resolve())
print('shape:', df.shape)
df.head()


D:\obi\熵学院\授课\机器学习-6小时\教学课程\day3\day3_cafe_sales.csv
shape: (200, 12)


,day,weather,weather_label,temp,is_weekend,promotion,quality,price,location,competitors,holiday,sales
0,1,1,多云,33.5,0,1,8.27,24.5,5.78,2,0,3249
1,2,2,阴,6.7,0,1,9.57,22.0,9.96,0,0,4700
2,3,0,晴,23.4,0,0,7.11,27.9,7.28,5,0,2071
3,4,2,阴,6.4,0,0,6.65,23.0,9.74,3,0,2370
4,5,0,晴,11.9,0,0,8.60,31.8,9.17,2,0,2847


In [2]:
# 准备特征和目标（沿用 Day 2 的思路）
feature_cols = ['price', 'promotion', 'is_weekend', 'temp', 'quality', 'competitors', 'holiday', 'location']
# 天气变成数字分：晴最好，大雨最差
weather_score_map = {'晴': 1.0, '多云': 0.8, '阴': 0.6, '小雨': 0.4, '大雨': 0.3}
df = df.copy()
df['weather_score'] = df['weather_label'].map(weather_score_map)

X = df[feature_cols + ['weather_score']].copy()
y = df['sales'].copy()

print('特征列:', list(X.columns))
print('样本数:', len(X))
X.head()


特征列: ['price', 'promotion', 'is_weekend', 'temp', 'quality', 'competitors', 'holiday', 'location', 'weather_score']
样本数: 200


,price,promotion,is_weekend,temp,quality,competitors,holiday,location,weather_score
0,24.5,1,0,33.5,8.27,2,0,5.78,0.8
1,22.0,1,0,6.7,9.57,0,0,9.96,0.6
2,27.9,0,0,23.4,7.11,5,0,7.28,1.0
3,23.0,0,0,6.4,6.65,3,0,9.74,0.6
4,31.8,0,0,11.9,8.60,2,0,9.17,1.0


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('训练集行数:', len(X_train))
print('测试集行数:', len(X_test))


训练集行数: 160
测试集行数: 40


## 1. 创建并训练随机森林

几个名字先混个脸熟：

- `n_estimators`：树的数量
- `max_depth`：每棵树大概能长多深
- `random_state`：固定随机过程，方便结果可复现


In [4]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)
print('随机森林训练完成')


随机森林训练完成


## 2. 看训练/测试表现（R²）

R² 越接近 1，通常表示拟合得越好。先会读数，公式细节后面再加深。


In [5]:
print('训练集 R²:', round(rf.score(X_train, y_train), 3))
print('测试集 R²:', round(rf.score(X_test, y_test), 3))


训练集 R²: 0.964
测试集 R²: 0.76


## 3. 和浅决策树简单对比


In [6]:
from sklearn.tree import DecisionTreeRegressor

tree = DecisionTreeRegressor(max_depth=3, random_state=42)
tree.fit(X_train, y_train)

print('决策树 测试 R²:', round(tree.score(X_test, y_test), 3))
print('随机森林 测试 R²:', round(rf.score(X_test, y_test), 3))


决策树 测试 R²: 0.522
随机森林 测试 R²: 0.76


## 4. 对测试集做预测


In [10]:
y_pred = rf.predict(X_test)
compare = pd.DataFrame({
    '真实营收': y_test.to_numpy()[:10],
    '预测营收': y_pred[:10].round(1),
    '相差': (y_pred[:10] - y_test.to_numpy()[:10]).round(1),
})
compare


,真实营收,预测营收,相差
0,2023,2114.3,91.3
1,5015,3939.2,-1075.8
2,2575,3106.9,531.9
3,3838,3776.8,-61.2
4,2396,2301.7,-94.3
5,2337,2595.7,258.7
6,2131,2222.7,91.7
7,2397,2405.7,8.7
8,3813,3349.9,-463.1
9,1893,1956.2,63.2


## 5. 可选：改一两个参数看一眼（不是系统调参）

今天主线参数先固定能跑通。有时间可以**只改一个**再跑，看测试 R² 有没有明显变化。

常见旋钮（先混个脸熟）：

| 参数 | 白话 |
| --- | --- |
| `n_estimators` | 森林里有多少棵树 |
| `max_depth` | 每棵树最多问几层 |
| `min_samples_leaf` | 叶子上至少要留几条样本（太小容易背题；今天可只知道名字） |

**不是**网格搜索；系统调参留给后面的课。


In [ ]:
# 可选：只改一个参数，重新 fit，看测试 R²
# 例 1：树的数量
rf2 = RandomForestRegressor(n_estimators=50, max_depth=8, random_state=42, n_jobs=-1)
rf2.fit(X_train, y_train)
print('n_estimators=50  测试 R²:', round(rf2.score(X_test, y_test), 3))

# 例 2：树深一点 / 不限制（可自行改 max_depth=4 或 None）
rf3 = RandomForestRegressor(n_estimators=100, max_depth=4, random_state=42, n_jobs=-1)
rf3.fit(X_train, y_train)
print('max_depth=4      测试 R²:', round(rf3.score(X_test, y_test), 3))

rf4 = RandomForestRegressor(n_estimators=200, max_depth=4, random_state=42, n_jobs=-1)
rf4.fit(X_train, y_train)
print('max_depth=4       测试 R²:', round(rf4.score(X_test, y_test), 3))


# 和主线 rf 对比
print('主线 n_estimators=100, max_depth=8 测试 R²:', round(rf.score(X_test, y_test), 3))


n_estimators=50  测试 R²: 0.755
max_depth=4      测试 R²: 0.697
主线 n_estimators=100, max_depth=8 测试 R²: 0.76
